In [3]:
import torch
import torch.nn as nn
import math


In [1]:
def generate_causal_mask(seq_len):
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask  # shape (seq_len, seq_len)

In [4]:
class MaskedSelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        assert self.head_dim * heads == embed_size

        self.values = nn.Linear(embed_size, embed_size)
        self.keys = nn.Linear(embed_size, embed_size)
        self.queries = nn.Linear(embed_size, embed_size)
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        N, seq_len, _ = x.shape

        values = self.values(x)
        keys = self.keys(x)
        queries = self.queries(x)

        # Split heads
        values = values.view(N, seq_len, self.heads, self.head_dim).transpose(1,2)
        keys = keys.view(N, seq_len, self.heads, self.head_dim).transpose(1,2)
        queries = queries.view(N, seq_len, self.heads, self.head_dim).transpose(1,2)

        energy = torch.matmul(queries, keys.transpose(-2,-1)) / math.sqrt(self.head_dim)

        # Apply causal mask
        mask = generate_causal_mask(seq_len).to(x.device)
        mask = mask.unsqueeze(0).unsqueeze(0)

        energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = torch.softmax(energy, dim=-1)

        out = torch.matmul(attention, values)
        out = out.transpose(1,2).contiguous().view(N, seq_len, self.embed_size)

        return self.fc_out(out)